# Singapore HDB resale market trends (last 5 years)

This notebook downloads the **HDB resale flat prices** dataset from the official **data.gov.sg dataset page** by following the page's **CSV download link**, geocodes each unique building address with **OneMap**, computes current price levels and 5-year changes, and exports two Singapore maps as **PDF**:

1. **Median resale price per square meter / square foot** by building  
2. **5-year percentage change** in median price per square meter by building

## Main assumptions

- The source dataset is the HDB dataset **“Resale flat prices based on registration date from Jan-2017 onwards”**.
- The analysis window is the **latest 5 years available in the dataset**.
- Because some buildings have sparse transactions, the 5-year change is estimated by comparing:
  - a **recent 12-month window**, and
  - the **same 12-month window shifted 5 years earlier**.
- One point on the map represents **one building**, identified primarily by **postal code** returned by OneMap.  
- Buildings that cannot be geocoded anymore (for example, demolished buildings) are excluded.

## Files produced

- `outputs/building_metrics.csv`
- `outputs/sg_hdb_psm_map.pdf`
- `outputs/sg_hdb_psf_map.pdf`
- `outputs/sg_hdb_5y_change_map.pdf`
- `outputs/sg_hdb_psm_interactive.html`
- `outputs/sg_hdb__5y_change_interactive.html`

## OPTIONAL
- I have downloaded all the geocodes for the buildings in the dataset (as of March 2026)
- You can use these geocodes directly
- If you download your own dataset that contains buildings not found in the dataset, you will need to run OneMap to collect the geocodes.

Place your OneMap API token:
- save it in a text file named `api.txt` in the same folder as this notebook


In [ ]:
# Optional: install packages the first time you run this notebook
# Uncomment and run if needed.
# %pip install pandas numpy requests tqdm matplotlib geopandas contextily shapely pyarrow folium branca


In [ ]:
from __future__ import annotations

import os
import time
import math
import json
import re
import html
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

import geopandas as gpd
from shapely.geometry import Point

try:
    import contextily as cx
    HAS_CONTEXTILY = True
except Exception:
    HAS_CONTEXTILY = False

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (9, 10)


In [ ]:
# ----------------------------
# Configuration
# ----------------------------

# 1. Data we want to analyze
# HDB_CSV_PATH: local path to the data we want to analyze
# Data Source: https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view; click "Download CSV".
# You can also view the column definitions using the link above.
# I have downaded the CSV file for you. The version is "Last updated: 15 Mar 2026, 02:10 SGT". You can use it directly.
# If you want to download the latest version on your own, please place the manually downloaded CSV in the same folder as this notebook, and change the file name accordingly.
HDB_CSV_PATH = "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"

# 2. Obtain geospatial data
# ONEMAP_SEARCH_API: url to OneMap.gov.sg. We need to use OneMap.gov.sg to find the latitute and longitude for each building so that we can display them on a map.
# To get your own API, you will need to register here: https://www.onemap.gov.sg/apidocs/register
# Then put your own API in the file 'api.txt' in the same folder as this notebook.
ONEMAP_SEARCH_API = "https://www.onemap.gov.sg/api/common/elastic/search"

# 3. Analysis Parameters
# 3a. LOOKBACK_YEARS: We are looking at "this year" as well as "5 years prior" for comparison
# 3b. WINDOW_MONTHS: For each year, we will look at a 12-month window 
# For example, if we are now in March 2026, we will look at the data periods April 2025 - March 2026, as well as April 2020 - March 2021
# While it may be more typical to look at "annual price change", we decide to look at "5-year price change" here, because:
# -- we want to observe longer price trends that are above and beyond temporary noises
# -- you can easily convert that number into annual price change later
LOOKBACK_YEARS = 5
WINDOW_MONTHS = 12

# 4. Directory and folders
# 4a. BASE_DIR: this is the base directory (the current directory of the Jupyter Notebook file)
# 4b. CACHE_DIR: this is the cache directory (to store the geospatial data so that we don't have to keep downloading it again and again)
# 4c: OUTPUT_DIR: this is the output directory to store our result files
# 4d: GEOCODE_BATCh_DIR: this is the directory to store the geocode downloaded in batches. 
# We will need to download the geocode for ~10k addresses, so we want to do it in small batches and store them locally in case of internet disruption.
BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "cache"
OUTPUT_DIR = BASE_DIR / "outputs"
GEOCODE_BATCH_DIR = CACHE_DIR / "geocode_batches"

CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
GEOCODE_BATCH_DIR.mkdir(exist_ok=True)

RAW_CACHE = CACHE_DIR / "hdb_resale_raw.csv"
GEOCODE_CACHE = CACHE_DIR / "address_geocodes.csv"

# 5. Geocoding is resumable.
# - BATCH_SIZE controls how many addresses are queried before writing a checkpoint.
# - MAX_BATCHES_PER_RUN lets you intentionally stop after N batches in one notebook run.
#   Set to None to process all remaining addresses.
BATCH_SIZE = 250
MAX_BATCHES_PER_RUN = None
SLEEP_SECONDS = 0.05

EXPECTED_COLUMNS = {
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease",
    "resale_price",
}

SG_BBOX = {
    "min_lon": 103.60,
    "max_lon": 104.05,
    "min_lat": 1.20,
    "max_lat": 1.48,
}


# Interactive HTML export settings (smaller files)
LIGHTWEIGHT_HTML = True
INCLUDE_STATION_MARKERS_IN_HTML = True    # keep MRT/LRT stations visible as dots in interactive HTML
RAIL_GEOMETRY_SIMPLIFY_METERS = 35        # simplify MRT/LRT line geometry for lighter HTML

DEDUPE_RAIL_LINES_FOR_DISPLAY = True      # collapse overlapping / near-parallel duplicate rail segments for cleaner display
RAIL_LINE_DEDUPE_BUFFER_METERS = 22     # lines closer than this and largely overlapping are treated as one visual line
BUILDING_POINT_RADIUS = 4
HTML_TOOLTIP_MODE = "compact"             # "compact" or "full"


In [ ]:
# # IGNORE IF YOU DON'T INTEND TO DOWNLOAD NEW DATA
# # YOU CAN USE THE DATA I SUPPLIED AND IT SHOULD WORK
# def load_onemap_token(token_file: str | Path = "api.txt") -> str:
#     token_path = Path(token_file)
#     if token_path.exists():
#         token = token_path.read_text(encoding="utf-8").strip()
#         if token:
#             return token

#     raise RuntimeError(
#         "OneMap API token not found. Save it in api.txt"
#     )

# ONEMAP_TOKEN = load_onemap_token("api.txt")
# print("Loaded OneMap token.")

## 1) Load the manually downloaded HDB resale CSV

This notebook now assumes you have already downloaded the official HDB resale CSV from data.gov.sg
and placed it in the same project folder as the notebook.

The loader below:
- reads the local CSV file directly
- validates that the expected HDB resale columns are present
- caches a copy under `cache/` for convenience

In [ ]:
def locate_hdb_csv(preferred_path: str | Path) -> Path:
    preferred = Path(preferred_path)
    if preferred.exists():
        return preferred
    
    raise FileNotFoundError(
        f"Could not find {preferred_path!r}. Put the downloaded CSV in the notebook folder "
        "or update HDB_CSV_PATH in the configuration cell."
    )

def load_hdb_resale_data_from_csv(csv_path: str | Path, use_cache: bool = False) -> pd.DataFrame:
    csv_file = locate_hdb_csv(csv_path)

    # Read all columns as strings first for safer validation, then convert selected fields later.
    df = pd.read_csv(csv_file)

    missing = EXPECTED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            "The CSV file does not look like the expected HDB resale dataset. "
            f"Missing columns: {sorted(missing)}"
        )

    if use_cache:
        df.to_csv(RAW_CACHE, index=False)

    print(f"Loaded local CSV: {csv_file}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df


In [ ]:
df_raw = load_hdb_resale_data_from_csv(HDB_CSV_PATH, use_cache=False)
df_raw.head()

The preprocessing below:
- clean up the input data to ensure proper formats. For example:
- convert numbers to numeric format
- use block and street_name to generate the full address
- calculate price per square meter (psm) and price per square feet (psf)
- drop any row with missing value.

In [ ]:
def preprocess_hdb_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["month"] = pd.to_datetime(df["month"].astype(str) + "-01", errors="coerce")
    df["floor_area_sqm"] = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
    df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")

    df["block"] = df["block"].astype(str).str.strip()
    df["street_name"] = df["street_name"].astype(str).str.strip()
    df["address"] = (
        df["block"] + " " + df["street_name"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    df["psm"] = df["resale_price"] / df["floor_area_sqm"]
    df["psf"] = df["psm"] / 10.76391041671

    df = df.dropna(subset=["month", "floor_area_sqm", "resale_price", "psm", "psf", "address"])
    df = df[df["floor_area_sqm"] > 0].copy()
    df = df.sort_values(["month", "town", "street_name", "block"]).reset_index(drop=True)
    return df


In [ ]:
df = preprocess_hdb_data(df_raw)
print(f"Cleaned rows: {len(df):,}")
df[["month", "town", "address", "floor_area_sqm", "resale_price", "psm", "psf"]].head()

Compute the prior-perid windows
- find the latest month in the data
- find the starting month
- compute the number of rows within this window

In [ ]:
latest_month = df["month"].max()
start_month = latest_month - pd.DateOffset(years=LOOKBACK_YEARS+1) + pd.offsets.MonthBegin(1)

df_5y = df[df["month"] >= start_month].copy()

print("Latest month in dataset:", latest_month.date())
print("5-year window starts:", start_month.date())
print("Rows in 5-year window:", f"{len(df_5y):,}")

## 2) Geocode each unique building with OneMap in resumable batches

This step is designed to be interruption-safe. It saves results after every batch, so if your internet drops, you can rerun the same cell and continue from where it stopped.

In [ ]:
def normalise_text(x: str) -> str:
    return str(x).upper().strip()

def score_onemap_result(result: dict, block: str, street_name: str, query: str) -> tuple:
    blk = normalise_text(result.get("BLK_NO", ""))
    road = normalise_text(result.get("ROAD_NAME", ""))
    postal = normalise_text(result.get("POSTAL", ""))
    address = normalise_text(result.get("ADDRESS", ""))

    block = normalise_text(block)
    street_name = normalise_text(street_name)
    query = normalise_text(query)

    score = 0
    if postal not in {"", "NIL"}:
        score += 5
    if blk == block:
        score += 5
    if street_name and street_name in road:
        score += 5
    if query and query in address:
        score += 2

    return (score,)

def geocode_one_address(session: requests.Session, address: str, block: str, street_name: str) -> dict:
    params = {
        "searchVal": address,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1,
    }
    r = session.get(ONEMAP_SEARCH_API, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()

    results = payload.get("results", []) or []
    if not results:
        return {
            "address": address,
            "postal": None,
            "latitude": np.nan,
            "longitude": np.nan,
            "geocode_status": "not_found",
        }

    best = sorted(
        results,
        key=lambda x: score_onemap_result(x, block=block, street_name=street_name, query=address),
        reverse=True,
    )[0]

    postal = best.get("POSTAL")
    lat = pd.to_numeric(best.get("LATITUDE"), errors="coerce")
    lon = pd.to_numeric(best.get("LONGITUDE"), errors="coerce")

    if pd.isna(lat) or pd.isna(lon) or str(postal).strip().upper() in {"", "NIL", "NONE"}:
        status = "invalid_or_missing_coords"
    else:
        status = "ok"

    return {
        "address": address,
        "postal": str(postal).strip() if postal is not None else None,
        "latitude": lat,
        "longitude": lon,
        "geocode_status": status,
    }

def load_geocode_cache() -> pd.DataFrame:
    if GEOCODE_CACHE.exists():
        cache = pd.read_csv(GEOCODE_CACHE, dtype={"postal": str})
    else:
        cache = pd.DataFrame(columns=["address", "postal", "latitude", "longitude", "geocode_status"])

    if not cache.empty:
        cache["latitude"] = pd.to_numeric(cache["latitude"], errors="coerce")
        cache["longitude"] = pd.to_numeric(cache["longitude"], errors="coerce")
        cache = cache.drop_duplicates(subset=["address"], keep="last")

    return cache

def save_geocode_batch(batch_df: pd.DataFrame, batch_number: int) -> Path:
    batch_path = GEOCODE_BATCH_DIR / f"geocode_batch_{batch_number:04d}.csv"
    batch_df.to_csv(batch_path, index=False)
    return batch_path

def update_master_geocode_cache(new_rows: list[dict]) -> pd.DataFrame:
    cache = load_geocode_cache()
    if new_rows:
        cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
        cache = cache.drop_duplicates(subset=["address"], keep="last")
        cache.to_csv(GEOCODE_CACHE, index=False)
    return cache

def geocode_addresses_in_batches(
    address_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE,
    max_batches_per_run: int | None = MAX_BATCHES_PER_RUN,
    sleep_seconds: float = SLEEP_SECONDS,
) -> pd.DataFrame:
    address_df = address_df[["address", "block", "street_name"]].drop_duplicates().copy()
    cache = load_geocode_cache()

    done_addresses = set(cache["address"].astype(str)) if not cache.empty else set()
    todo = address_df[~address_df["address"].astype(str).isin(done_addresses)].copy().reset_index(drop=True)

    print(f"Already cached: {len(done_addresses):,}")
    print(f"Remaining to geocode: {len(todo):,}")

    if todo.empty:
        return cache

    existing_batch_numbers = []
    for p in GEOCODE_BATCH_DIR.glob("geocode_batch_*.csv"):
        m = re.search(r"(\d+)", p.stem)
        if m:
            existing_batch_numbers.append(int(m.group(1)))
    next_batch_number = max(existing_batch_numbers, default=0) + 1

    session = requests.Session()
    session.headers.update({"Authorization": ONEMAP_TOKEN})

    batches_completed_this_run = 0
    n_batches = math.ceil(len(todo) / batch_size)

    for batch_idx in range(n_batches):
        if max_batches_per_run is not None and batches_completed_this_run >= max_batches_per_run:
            print(f"Stopped after {batches_completed_this_run} batch(es) this run. Re-run the cell to continue.")
            break

        start = batch_idx * batch_size
        stop = min((batch_idx + 1) * batch_size, len(todo))
        batch = todo.iloc[start:stop].copy()

        new_rows = []
        for row in tqdm(batch.itertuples(index=False), total=len(batch), desc=f"Geocode batch {next_batch_number}"):
            try:
                item = geocode_one_address(
                    session=session,
                    address=row.address,
                    block=row.block,
                    street_name=row.street_name,
                )
            except Exception as e:
                item = {
                    "address": row.address,
                    "postal": None,
                    "latitude": np.nan,
                    "longitude": np.nan,
                    "geocode_status": f"error: {e}",
                }
            new_rows.append(item)
            time.sleep(sleep_seconds)

        batch_df = pd.DataFrame(new_rows)
        batch_path = save_geocode_batch(batch_df, next_batch_number)
        cache = update_master_geocode_cache(new_rows)

        batches_completed_this_run += 1
        print(
            f"Saved batch {next_batch_number} to {batch_path}. "
            f"Master cache now has {len(cache):,} unique addresses."
        )
        next_batch_number += 1

    cache = load_geocode_cache()
    return cache


In [ ]:
unique_addresses = df_5y[["address", "block", "street_name"]].drop_duplicates()

# we will reuse the geocodes if we can
# if there are new addresses that are not found in the available geocodes, then we download them
geocodes_avail = pd.read_csv('cache/address_geocodes.csv')
unique_addresses_missing = df_5y[~df_5y["address"].isin(geocodes_avail["address"])][["address", "block", "street_name"]].drop_duplicates()
if len(unique_addresses_missing) > 0:
    geocodes_new = geocode_addresses_in_batches(
    unique_addresses_missing,
    batch_size=BATCH_SIZE,
    max_batches_per_run=MAX_BATCHES_PER_RUN,
    sleep_seconds=SLEEP_SECONDS,
    )

geocodes = pd.read_csv('cache/address_geocodes.csv')

geocodes["usable"] = (
    geocodes["geocode_status"].eq("ok")
    & geocodes["latitude"].between(SG_BBOX["min_lat"], SG_BBOX["max_lat"])
    & geocodes["longitude"].between(SG_BBOX["min_lon"], SG_BBOX["max_lon"])
)

print(geocodes["geocode_status"].value_counts(dropna=False).head(10))
print("Usable geocodes:", int(geocodes["usable"].sum()), "out of", len(geocodes))
geocodes.head()

## 3) Merge transactions with geocodes and exclude missing/demolished buildings

In [ ]:
df_geo = df_5y.merge(
    geocodes[["address", "postal", "latitude", "longitude", "geocode_status", "usable"]],
    on="address",
    how="left",
)

df_geo = df_geo[df_geo["usable"]].copy()
df_geo = df_geo.dropna(subset=["postal", "latitude", "longitude"])

print("Geocoded transaction rows:", f"{len(df_geo):,}")
print("Unique buildings:", f"{df_geo['postal'].nunique():,}")
df_geo.head()

## 4) Compute building-level current price and 5-year change

### Current price level
For each building, use the **median** transaction price per square meter / square foot over the **latest 12 months**.

### 5-year change
For each building, compare:
- the building's median `psm` in the **latest 12 months**
- with the building's median `psm` in the **same 12-month window 5 years earlier**

This avoids unstable year-by-year changes for buildings with very few transactions.

In [ ]:
analysis_end = df_geo["month"].max()
recent_start = analysis_end - pd.DateOffset(months=WINDOW_MONTHS - 1)
base_end = analysis_end - pd.DateOffset(years=LOOKBACK_YEARS)
base_start = recent_start - pd.DateOffset(years=LOOKBACK_YEARS)

print("Recent window:", recent_start.date(), "to", analysis_end.date())
print("Base window:  ", base_start.date(), "to", base_end.date())

In [ ]:
recent_mask = (df_geo["month"] >= recent_start) & (df_geo["month"] <= analysis_end)
base_mask = (df_geo["month"] >= base_start) & (df_geo["month"] <= base_end)

recent = (
    df_geo.loc[recent_mask]
    .groupby("postal", as_index=False)
    .agg(
        current_psm=("psm", "median"),
        current_psf=("psf", "median"),
        recent_txn_count=("psm", "size"),
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        address=("address", "first"),
        town=("town", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
    )
)

base = (
    df_geo.loc[base_mask]
    .groupby("postal", as_index=False)
    .agg(
        base_psm=("psm", "median"),
        base_psf=("psf", "median"),
        base_txn_count=("psm", "size"),
    )
)

building_metrics = recent.merge(base, on="postal", how="left")
building_metrics["pct_change_psm_5y"] = round((
    (building_metrics["current_psm"] / building_metrics["base_psm"]) - 1.0
) * 100.0, 2)
building_metrics["pct_change_psf_5y"] = round((
    (building_metrics["current_psf"] / building_metrics["base_psf"]) - 1.0
) * 100.0, 2)

building_metrics = building_metrics.sort_values("current_psm", ascending=False).reset_index(drop=True)
building_metrics.to_csv(OUTPUT_DIR / "building_metrics.csv", index=False)

print(building_metrics.shape)
building_metrics.head()

In [ ]:
# round to 2 decimals
building_metrics[["current_psm", "current_psf", "base_psm", "base_psf"]] = (
    building_metrics[["current_psm", "current_psf", "base_psm", "base_psf"]].round(2)
)

In [ ]:
coverage = pd.Series({
    "buildings_in_recent_window": recent["postal"].nunique(),
    "buildings_with_5y_change": building_metrics["pct_change_psm_5y"].notna().sum(),
})
coverage

## 5A) Download and Overlay MRT / LRT Line and Station Layers

To make it easier to identify which HDB flats are located closer to rail transit lines, this section will load additional official public data layers for Singapore:

- **URA Master Plan 2019 Rail Line layer**: MRT / LRT / Railway lines
- **URA Master Plan 2019 Rail Station layer**: MRT / LRT station outlines

The code will prioritize using the local cache located in `cache/reference_layers/`; if the cache does not exist, it will automatically download the data from data.gov.sg and save it locally.
In the event of a network interruption, as long as the cache files have already been generated, they will not need to be re-downloaded during subsequent runs.

In [ ]:
REFERENCE_DIR = CACHE_DIR / "reference_layers"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

URA_RAIL_LINE_DATASET_ID = "d_222bfc84eb86c7c11994d02f8939da8d"
URA_RAIL_STATION_DATASET_ID = "d_8d886e3a83934d7447acdf5bc6959999"

URA_RAIL_LINE_LOCAL = REFERENCE_DIR / "ura_master_plan_2019_rail_line.geojson"
URA_RAIL_STATION_LOCAL = REFERENCE_DIR / "ura_master_plan_2019_rail_station.geojson"


In [ ]:
def poll_download_dataset(dataset_id: str, file_path: Path, timeout: int = 120) -> Path:
    """
    Use the official data.gov.sg public API to request a downloadable file URL,
    then save the dataset locally so future notebook runs can reuse it.
    """
    if file_path.exists():
        return file_path

    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    poll_resp = requests.get(poll_url, timeout=timeout)
    poll_resp.raise_for_status()
    poll_json = poll_resp.json()

    if "data" not in poll_json or "url" not in poll_json["data"]:
        raise RuntimeError(f"Unexpected poll-download response for {dataset_id}: {poll_json}")

    download_url = poll_json["data"]["url"]
    file_resp = requests.get(download_url, timeout=timeout)
    file_resp.raise_for_status()

    file_path.write_bytes(file_resp.content)
    return file_path


def load_reference_layer(dataset_id: str, local_path: Path) -> gpd.GeoDataFrame:
    local_path = poll_download_dataset(dataset_id, local_path)
    gdf = gpd.read_file(local_path)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)
    return gdf


In [ ]:
gdf_rail_lines = load_reference_layer(URA_RAIL_LINE_DATASET_ID, URA_RAIL_LINE_LOCAL)
gdf_rail_stations_raw = load_reference_layer(URA_RAIL_STATION_DATASET_ID, URA_RAIL_STATION_LOCAL)

# Convert station polygons into points for plotting / nearest-station calculations
gdf_rail_station_points = gdf_rail_stations_raw.copy()
gdf_rail_station_points["geometry"] = gdf_rail_station_points.geometry.centroid

station_name_col = "NAME" if "NAME" in gdf_rail_station_points.columns else None
if station_name_col is None:
    gdf_rail_station_points["NAME"] = "MRT/LRT station"

print("Rail lines:", len(gdf_rail_lines))
print("Rail stations:", len(gdf_rail_station_points))
gdf_rail_station_points.head()


In [ ]:
def build_geodf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    out = df.copy()
    geometry = [Point(xy) for xy in zip(out["longitude"], out["latitude"])]
    gdf = gpd.GeoDataFrame(out, geometry=geometry, crs="EPSG:4326")
    return gdf


def attach_nearest_station(
    gdf_buildings: gpd.GeoDataFrame,
    gdf_station_points: gpd.GeoDataFrame,
    station_name_col: str = "NAME",
) -> gpd.GeoDataFrame:
    """
    Add the nearest MRT/LRT station name and straight-line distance (meters)
    to each building. This is helpful for the interactive tooltip.
    """
    buildings = gdf_buildings.to_crs(epsg=3857).copy()
    stations = gdf_station_points.to_crs(epsg=3857).copy()

    bxy = np.column_stack([buildings.geometry.x.to_numpy(), buildings.geometry.y.to_numpy()])
    sxy = np.column_stack([stations.geometry.x.to_numpy(), stations.geometry.y.to_numpy()])

    # 9k buildings x ~200 stations is still small enough for a dense distance matrix.
    dx = bxy[:, None, 0] - sxy[None, :, 0]
    dy = bxy[:, None, 1] - sxy[None, :, 1]
    dist2 = dx * dx + dy * dy

    nearest_idx = dist2.argmin(axis=1)
    nearest_dist_m = np.sqrt(dist2[np.arange(len(buildings)), nearest_idx]).round(0)

    buildings["nearest_station"] = stations.iloc[nearest_idx][station_name_col].to_numpy()
    buildings["nearest_station_distance_m"] = nearest_dist_m

    return buildings.to_crs(epsg=4326)


In [ ]:
gdf_buildings = build_geodf(building_metrics)
gdf_buildings = attach_nearest_station(
    gdf_buildings,
    gdf_rail_station_points,
    station_name_col="NAME",
)
gdf_buildings.head()


## 5B) Simplify Parallel rail lines

In [ ]:

def dedupe_parallel_rail_lines(
    rail_lines: gpd.GeoDataFrame | None,
    overlap_buffer_m: float = 22,
    overlap_ratio_threshold: float = 0.60,
) -> gpd.GeoDataFrame | None:
    """
    Remove visually redundant rail segments when the source layer contains
    two near-parallel tracks between the same station pair.

    Strategy:
    - project to a metric CRS
    - compare buffered overlap against already-kept segments
    - keep only one representative segment when two lines are almost the same
      for display purposes
    """
    if rail_lines is None or rail_lines.empty:
        return rail_lines

    lines = rail_lines.copy()
    crs_original = lines.crs
    try:
        metric = lines.to_crs(epsg=3414)
        metric = metric.loc[~metric.geometry.is_empty & metric.geometry.notna()].copy()
        if metric.empty:
            return rail_lines

        metric["geometry"] = metric.geometry.line_merge() if hasattr(metric.geometry, "line_merge") else metric.geometry
        kept_idx = []
        kept_buffers = []

        for idx, geom in metric.geometry.items():
            if geom is None or geom.is_empty:
                continue

            geom_buf = geom.buffer(overlap_buffer_m, cap_style=2, join_style=2)
            is_duplicate = False

            for kept_buf in kept_buffers:
                inter_area = geom_buf.intersection(kept_buf).area
                min_area = min(geom_buf.area, kept_buf.area)
                if min_area > 0 and (inter_area / min_area) >= overlap_ratio_threshold:
                    is_duplicate = True
                    break

            if not is_duplicate:
                kept_idx.append(idx)
                kept_buffers.append(geom_buf)

        if not kept_idx:
            return rail_lines

        deduped = metric.loc[kept_idx].copy()
        return deduped.to_crs(crs_original)
    except Exception:
        return rail_lines


def simplify_lines_for_web(
    rail_lines: gpd.GeoDataFrame | None,
    tolerance_m: float = 35,
    dedupe_parallel: bool = True,
    dedupe_buffer_m: float = 22,
) -> gpd.GeoDataFrame | None:
    if rail_lines is None or rail_lines.empty:
        return rail_lines

    lines = rail_lines.copy()
    crs_original = lines.crs
    try:
        metric = lines.to_crs(epsg=3414)  # Singapore projected CRS in meters
        if tolerance_m > 0:
            metric["geometry"] = metric.geometry.simplify(tolerance=tolerance_m, preserve_topology=True)
        lines_out = metric.to_crs(crs_original)
    except Exception:
        lines_out = rail_lines

    if dedupe_parallel:
        lines_out = dedupe_parallel_rail_lines(
            lines_out,
            overlap_buffer_m=dedupe_buffer_m,
            overlap_ratio_threshold=0.60,
        )

    return lines_out


## 6A) Plot Singapore maps with clipped color scales

To stop extreme outliers from dominating the colors:
- for price level, clip the scale to the **2nd and 98th percentiles**
- for percentage change, use a **symmetric clipped scale** around zero

The maps below attempt to add a light basemap using `contextily`.  
If tile fetching fails, the scatter map still renders and can still be exported to PDF.

In [ ]:
def percentile_clip(series: pd.Series, low: float = 0.1, high: float = 0.9) -> tuple[float, float]:
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return (np.nan, np.nan)
    return (float(s.quantile(low)), float(s.quantile(high)))

def plot_metric_map(
    gdf: gpd.GeoDataFrame,
    metric_col: str,
    title: str,
    output_pdf: Path,
    cmap: str = "viridis",
    clip_quantiles: tuple[float, float] = (0.02, 0.98),
    center_zero: bool = False,
    marker_size: int = 22,
    rail_lines: gpd.GeoDataFrame | None = None,
    station_points: gpd.GeoDataFrame | None = None,
    legend_label: str | None = None,
    dedupe_rail_lines: bool = True,
    dedupe_rail_buffer_m: float = 22,
):
    data = gdf.dropna(subset=[metric_col]).copy()
    if data.empty:
        raise ValueError(f"No data available for {metric_col}")

    q_low, q_high = percentile_clip(data[metric_col], *clip_quantiles)
    data["metric_clipped"] = data[metric_col].clip(q_low, q_high)

    data_3857 = data.to_crs(epsg=3857)

    rail_lines_3857 = None
    if rail_lines is not None and not rail_lines.empty:
        rail_lines_clean = simplify_lines_for_web(
            rail_lines,
            tolerance_m=0,
            dedupe_parallel=dedupe_rail_lines,
            dedupe_buffer_m=dedupe_rail_buffer_m,
        )
        rail_lines_3857 = rail_lines_clean.to_crs(epsg=3857)

    station_points_3857 = None
    if station_points is not None and not station_points.empty:
        station_points_3857 = station_points.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(9, 10))

    if center_zero:
        bound = max(abs(q_low), abs(q_high))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0.0, vmax=bound)
    else:
        norm = Normalize(vmin=q_low, vmax=q_high)

    if rail_lines_3857 is not None:
        rail_lines_3857.plot(
            ax=ax,
            color="#444444",
            linewidth=1.0,
            alpha=0.45,
            zorder=1,
        )

    if station_points_3857 is not None:
        station_points_3857.plot(
            ax=ax,
            color="black",
            markersize=5,
            alpha=0.35,
            zorder=2,
        )

    data_3857.plot(
        ax=ax,
        column="metric_clipped",
        cmap=cmap,
        markersize=marker_size,
        alpha=0.9,
        legend=True,
        norm=norm,
        legend_kwds={"shrink": 0.35, "label": legend_label or metric_col},
        zorder=3,
    )

    if HAS_CONTEXTILY:
        try:
            cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False)
        except Exception as e:
            print(f"Basemap fetch failed, continuing without tiles: {e}")

    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

    # Zoom to the buildings with a small margin
    xmin, ymin, xmax, ymax = data_3857.total_bounds
    xpad = (xmax - xmin) * 0.05
    ypad = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    plt.tight_layout()
    plt.savefig(output_pdf, bbox_inches="tight")
    plt.show()
       


In [ ]:
plot_metric_map(
    gdf_buildings,
    metric_col="current_psm",
    title="Singapore HDB resale market — \n median price per sqm by building (latest 12 months)",
    output_pdf=OUTPUT_DIR / "sg_hdb_psm_map.pdf",
    cmap="RdYlBu_r",
    clip_quantiles=(0.15, 0.85),
    center_zero=False,
    marker_size=1,
    rail_lines=gdf_rail_lines,
    station_points=gdf_rail_station_points,
    legend_label="SGD per square meter",
    dedupe_rail_lines=False,
    dedupe_rail_buffer_m=RAIL_LINE_DEDUPE_BUFFER_METERS,
)


In [ ]:
plot_metric_map(
    gdf_buildings,
    metric_col="current_psf",
    title="Singapore HDB resale market — \n median price per sqf by building (latest 12 months)",
    output_pdf=OUTPUT_DIR / "sg_hdb_psf_map.pdf",
    cmap="RdYlBu_r",
    clip_quantiles=(0.15, 0.85),
    center_zero=False,
    marker_size=1,
    rail_lines=gdf_rail_lines,
    station_points=gdf_rail_station_points,
    legend_label="SGD per square feet",
    dedupe_rail_lines=False,
    dedupe_rail_buffer_m=RAIL_LINE_DEDUPE_BUFFER_METERS,
)


In [ ]:
plot_metric_map(
    gdf_buildings,
    metric_col="pct_change_psm_5y",
    title="Singapore HDB resale market — \n 5-year % change in median price per sqm by building",
    output_pdf=OUTPUT_DIR / "sg_hdb_5y_change_map.pdf",
    cmap="RdYlBu_r",
    clip_quantiles=(0.15, 0.85),
    center_zero=False,
    marker_size=1,
    rail_lines=gdf_rail_lines,
    station_points=gdf_rail_station_points,
    legend_label="Change over 5 Years (%)",
    dedupe_rail_lines=False,
    dedupe_rail_buffer_m=RAIL_LINE_DEDUPE_BUFFER_METERS,
)


## 6B) Generate Interactive Maps (Zoomable; Hover to View Building Information)

Two additional **interactive HTML maps** will be generated below:

1.  Color-coded by the building's median **price per square meter** over the past 12 months.
2.  Color-coded by the building's **5-year price change rate**.

Features include:
-   Hovering / clicking to view the building's address, postal code, district, price, transaction count, and the name and straight-line distance of the nearest MRT/LRT station.
-   Zooming and panning capabilities.
-   Overlay of MRT/LRT lines and stations.
-   Can be opened directly in a web browser or displayed within a Jupyter environment.


In [ ]:

import folium
from branca.colormap import LinearColormap


def fmt_text(value, default: str = "N/A") -> str:
    if pd.isna(value):
        return default
    text = str(value).strip()
    return default if text == "" else text


def fmt_number(value, digits: int = 0, suffix: str = "", default: str = "N/A") -> str:
    if pd.isna(value):
        return default
    return f"{value:,.{digits}f}{suffix}"


def fmt_int(value, default: str = "N/A") -> str:
    if pd.isna(value):
        return default
    return f"{int(round(value)):,}"


def build_tooltip_fields(row: pd.Series, mode: str = "compact") -> list[tuple[str, str]]:
    pct_text = fmt_number(row.get("pct_change_psm_5y"), digits=1, suffix="%", default="N/A")
    dist_text = fmt_number(row.get("nearest_station_distance_m"), digits=0, suffix=" m", default="N/A")
    current_psm_text = fmt_number(row.get("current_psm"), digits=0, suffix=" SGD", default="N/A")
    current_psf_text = fmt_number(row.get("current_psf"), digits=0, suffix=" SGD", default="N/A")

    compact_fields = [
        ("Address", fmt_text(row.get("address"))),
        ("Town", fmt_text(row.get("town"))),
        ("PSM (12m)", current_psm_text),
        ("PSF (12m)", current_psf_text),
        ("5Y change", pct_text),
        ("Nearest MRT/LRT", fmt_text(row.get("nearest_station"))),
        ("Distance", dist_text),
    ]
    if mode == "compact":
        return compact_fields

    return compact_fields[:2] + [
        ("Postal", fmt_text(row.get("postal"))),
        ("Transactions (12m)", fmt_int(row.get("recent_txn_count"), default="N/A")),
        ("Transactions (base)", fmt_int(row.get("base_txn_count"), default="N/A")),
    ] + compact_fields[2:]


def make_tooltip_html(row: pd.Series, mode: str = "compact") -> str:
    fields = build_tooltip_fields(row, mode=mode)
    return "<br>".join(f"<b>{label}:</b> {value}" for label, value in fields)


def make_interactive_metric_map(
    gdf: gpd.GeoDataFrame,
    metric_col: str,
    map_title: str,
    output_html: Path,
    rail_lines: gpd.GeoDataFrame | None = None,
    station_points: gpd.GeoDataFrame | None = None,
    clip_quantiles: tuple[float, float] = (0.02, 0.98),
    color_scale: list[str] | None = None,
    include_station_markers: bool = True,
    simplify_rail_tolerance_m: float = 0,
    dedupe_rail_lines: bool = True,
    dedupe_rail_buffer_m: float = 22,
    tooltip_mode: str = "compact",
    point_radius: float = 4,
):
    data = gdf.dropna(subset=[metric_col]).copy()
    if data.empty:
        raise ValueError(f"No data available for {metric_col}")

    # Remove unused columns before serialising to HTML to keep files smaller
    keep_cols = [
        "address", "postal", "town", "current_psm", "current_psf",
        "pct_change_psm_5y", "recent_txn_count", "base_txn_count",
        "nearest_station", "nearest_station_distance_m", "latitude", "longitude", "geometry",
        metric_col,
    ]
    keep_cols = [c for c in keep_cols if c in data.columns]
    keep_cols = list(dict.fromkeys(keep_cols))
    data = data[keep_cols].copy()

    q_low, q_high = percentile_clip(data[metric_col], *clip_quantiles)
    data["metric_clipped"] = data[metric_col].clip(q_low, q_high)

    if color_scale is None:
        color_scale = ["#4575b4", "#91bfdb", "#ffffbf", "#fc8d59", "#d73027"]

    sg_center = [data["latitude"].median(), data["longitude"].median()]
    m = folium.Map(location=sg_center, zoom_start=11, tiles="CartoDB positron", control_scale=True)

    title_html = f"""
         <div style="position: fixed;
                     top: 10px; left: 50px; width: 420px; z-index:9999;
                     background-color: white; padding: 10px 12px; border:2px solid #666;
                     font-size:14px;">
         <b>{map_title}</b>
         </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))

    colormap = LinearColormap(
        colors=color_scale,
        vmin=q_low,
        vmax=q_high,
        caption=map_title,
    )
    colormap.add_to(m)

    lines_for_web = simplify_lines_for_web(
        rail_lines,
        tolerance_m=simplify_rail_tolerance_m,
        dedupe_parallel=dedupe_rail_lines,
        dedupe_buffer_m=dedupe_rail_buffer_m,
    )
    if lines_for_web is not None and not lines_for_web.empty:
        folium.GeoJson(
            lines_for_web.to_json(),
            name="MRT/LRT lines",
            style_function=lambda feature: {
                "color": "#444444",
                "weight": 2,
                "opacity": 0.55,
            },
            smooth_factor=2,
        ).add_to(m)

    if include_station_markers and station_points is not None and not station_points.empty:
        station_group = folium.FeatureGroup(name="MRT/LRT stations", show=True)
        station_min = station_points[["geometry", "NAME"]].copy()
        for _, row in station_min.iterrows():
            folium.CircleMarker(
                location=[round(row.geometry.y, 5), round(row.geometry.x, 5)],
                radius=2,
                color="black",
                weight=1,
                fill=True,
                fill_opacity=0.55,
                tooltip=str(row.get("NAME", "MRT/LRT station")),
            ).add_to(station_group)
        station_group.add_to(m)

    building_group = folium.FeatureGroup(name="HDB buildings", show=True)
    for _, row in data.iterrows():
        value = row["metric_clipped"]
        folium.CircleMarker(
            location=[round(row.geometry.y, 5), round(row.geometry.x, 5)],
            radius=point_radius,
            color=colormap(value),
            weight=0.6,
            fill=True,
            fill_color=colormap(value),
            fill_opacity=0.85,
            tooltip=folium.Tooltip(make_tooltip_html(row, mode=tooltip_mode), sticky=True),
        ).add_to(building_group)
    building_group.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(str(output_html))
    return m

In [ ]:
interactive_psm_map = make_interactive_metric_map(
    gdf_buildings,
    metric_col="current_psm",
    map_title="Singapore HDB resale market — \n median price per sqm by building (latest 12 months) - Interactive",
    output_html=OUTPUT_DIR / "sg_hdb_psm_interactive.html",
    rail_lines=gdf_rail_lines,
    station_points=gdf_rail_station_points,
    clip_quantiles=(0.15, 0.85),
    include_station_markers=INCLUDE_STATION_MARKERS_IN_HTML,
    simplify_rail_tolerance_m=RAIL_GEOMETRY_SIMPLIFY_METERS if LIGHTWEIGHT_HTML else 0,
    dedupe_rail_lines=DEDUPE_RAIL_LINES_FOR_DISPLAY,
    dedupe_rail_buffer_m=RAIL_LINE_DEDUPE_BUFFER_METERS,
    tooltip_mode=HTML_TOOLTIP_MODE,
    point_radius=BUILDING_POINT_RADIUS,
)

In [ ]:
interactive_psm_map

In [ ]:
interactive_change_map = make_interactive_metric_map(
    gdf_buildings,
    metric_col="pct_change_psm_5y",
    map_title="Singapore HDB resale market — \n 5-year % change in median price per sqm by building - Interactive",
    output_html=OUTPUT_DIR / "sg_hdb_5y_change_interactive.html",
    rail_lines=gdf_rail_lines,
    station_points=gdf_rail_station_points,
    clip_quantiles=(0.15, 0.85),
    include_station_markers=INCLUDE_STATION_MARKERS_IN_HTML,
    simplify_rail_tolerance_m=RAIL_GEOMETRY_SIMPLIFY_METERS if LIGHTWEIGHT_HTML else 0,
    dedupe_rail_lines=DEDUPE_RAIL_LINES_FOR_DISPLAY,
    dedupe_rail_buffer_m=RAIL_LINE_DEDUPE_BUFFER_METERS,
    tooltip_mode=HTML_TOOLTIP_MODE,
    point_radius=BUILDING_POINT_RADIUS,
)

In [ ]:
interactive_change_map

In [ ]:

print("Interactive HTML files written to:")
print(" -", OUTPUT_DIR / "sg_hdb_psm_interactive.html")
print(" -", OUTPUT_DIR / "sg_hdb_5y_change_interactive.html")

## 6C) Quick checks and summaries

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "current_psm",
        "current_psf",
        "pct_change_psm_5y",
        "pct_change_psf_5y",
    ],
    "count": [
        building_metrics["current_psm"].notna().sum(),
        building_metrics["current_psf"].notna().sum(),
        building_metrics["pct_change_psm_5y"].notna().sum(),
        building_metrics["pct_change_psf_5y"].notna().sum(),
    ],
    "p02": [
        building_metrics["current_psm"].quantile(0.02),
        building_metrics["current_psf"].quantile(0.02),
        building_metrics["pct_change_psm_5y"].quantile(0.02),
        building_metrics["pct_change_psf_5y"].quantile(0.02),
    ],
    "median": [
        building_metrics["current_psm"].median(),
        building_metrics["current_psf"].median(),
        building_metrics["pct_change_psm_5y"].median(),
        building_metrics["pct_change_psf_5y"].median(),
    ],
    "p98": [
        building_metrics["current_psm"].quantile(0.98),
        building_metrics["current_psf"].quantile(0.98),
        building_metrics["pct_change_psm_5y"].quantile(0.98),
        building_metrics["pct_change_psf_5y"].quantile(0.98),
    ],
})
summary

In [ ]:
# Top buildings by latest median PSM
building_metrics.nlargest(20, "current_psm")[[
    "postal", "address", "town", "current_psm", "current_psf", "recent_txn_count"
]]

In [ ]:
# Top and bottom 20 buildings by 5-year % change
building_metrics.dropna(subset=["pct_change_psm_5y"]).sort_values("pct_change_psm_5y", ascending=False)[[
    "postal", "address", "town", "pct_change_psm_5y", "current_psm", "base_psm"
]].head(20)

## 7) Notes for interpretation

- These are **transaction-based medians**, not hedonic-model estimates.
- Buildings with very few transactions can still be noisier than high-volume buildings, even after using 12-month windows.
- Geocoding is based on **current OneMap search results**. Buildings that are no longer returned by OneMap are excluded.
- The MRT / LRT overlays come from official **URA Master Plan 2019 Rail Line / Rail Station** layers downloaded from data.gov.sg.
- The nearest-station field in the interactive map is based on **straight-line distance** from each building point to the station point, not walking distance.
- If you want a stricter building identity rule, you can require an exact postal code and inspect ambiguous addresses manually.
- You can change `WINDOW_MONTHS` from `12` to `18` or `24` if you want more stable building-level estimates.
